# MiniGPT reads further: follow along

This notebook goes with my post [MiniGPT (Part 7)](https://haddley.github.io/posts/minigpt6/). It counts how fast attention's matches grow, checks that the chunked sliding window gives exactly the right answers, and races full attention against the window for memory.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part7-sliding-window")
sys.path.insert(0, os.path.abspath("../part5-modern-block"))

## 1. Counting the matches

Every working card matches its query against every position up to its own. Each doubling of the row roughly quadruples the matches. With a 256-position window, they grow only in step with the row.

In [ ]:
def full_matches(T):
    return T * (T + 1) // 2

def window_matches(T, w=256):
    return sum(min(i + 1, w) for i in range(T))

for T in (4, 8, 1024, 2048, 4096):
    print(f"row {T:>5}: full attention {full_matches(T):>12,} matches, 256-position window {window_matches(T):>10,}")

## 2. The chunked window gives the right answers

With a window as long as the whole row, `chunked_swa` must give exactly full attention's numbers. With a real window, it must match the masked version.

In [ ]:
import mlx.core as mx
from model_llama import chunked_swa, sliding_window_mask

mx.random.seed(0)
B, H, T, D = 1, 2, 512, 64
q, k, v = (mx.random.normal((B, H, T, D)) for _ in range(3))
scale = D ** -0.5

full = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale, mask="causal")
print("window = whole row, biggest difference from full attention:",
      mx.abs(chunked_swa(q, k, v, T, scale) - full).max().item())

masked = mx.fast.scaled_dot_product_attention(q, k, v, scale=scale, mask=sliding_window_mask(T, 64, q.dtype))
print("window = 64, biggest difference from the masked version:   ",
      mx.abs(chunked_swa(q, k, v, 64, scale) - masked).max().item())

## 3. A short memory race

The same few training steps as `mem_sweep.py`, at three row lengths. The full sweep, up to 16,384 positions, is `python mem_sweep.py`, and needs a 64 GB Mac.

In [ ]:
from mem_sweep import run as race
for T in (512, 1024, 2048):
    for mode in ("full", "naive", "chunked"):
        peak, dt = race(T, mode)
        print(f"row {T:>5}  {mode:8}  most memory {peak:6.2f} GB   {dt * 1000:7.1f} ms per step")